In [1]:
!pip install google-cloud-documentai
!pip install vertexai
!pip install spacy
!pip install nltk
!pip install google-api-core
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------------ --------------------------- 3.9/12.8 MB 23.5 MB/s eta 0:00:01
     ----------------------- ---------------- 7.6/12.8 MB 20.4 MB/s eta 0:00:01
     ------------------------------- ------- 10.2/12.8 MB 18.2 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 16.8 MB/s eta 0:00:01
     --------------------------------------- 12.8/12.8 MB 16.1 MB/s eta 0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:

import logging
import json
import re
import os
import spacy
import dateutil.parser
from typing import Optional, Dict, Union, List
from google.api_core.client_options import ClientOptions
from google.cloud import documentai
from vertexai.generative_models import GenerativeModel
import vertexai

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

class AdvancedTextPreprocessor:
    def __init__(self):
        self.logger = logging.getLogger(__name__)
        try:
            self.nlp = spacy.load('en_core_web_sm')
        except OSError:
            self.logger.error("Spacy model not found. Installing...")
            os.system("python -m spacy download en_core_web_sm")
            self.nlp = spacy.load('en_core_web_sm')
            
        self.patterns = {
            'html': r'<[^>]+>',
            'emails': r'\S+@\S+',
            'urls': r'http\S+',
            'special_chars': r'[^\w\s.,!?;:()\'"/-]',
            'multiple_spaces': r'\s+',
            'bullet_points': r'[•●■◆▪️]+\s*',
            'multiple_newlines': r'\n+'
        }

    def preprocess(self, text: str, debug: bool = False) -> str:
        try:
            if debug:
                print("Original text:", text[:200] + "...")
            
            # Basic cleaning
            text = self._remove_patterns(text)
            
            # Date standardization
            text = self._standardize_dates(text)
            
            # Entity processing
            text = self._process_entities(text)
            
            # Final whitespace normalization
            text = self._normalize_whitespace(text)
            
            if debug:
                print("\nFinal processed text:", text[:200] + "...")
            
            return text.strip()
        except Exception as e:
            self.logger.error(f"Error during preprocessing: {str(e)}")
            raise

    def _remove_patterns(self, text: str) -> str:
        for pattern_name, pattern in self.patterns.items():
            text = re.sub(pattern, ' ', text)
        return text

    def _standardize_dates(self, text: str) -> str:
        doc = self.nlp(text)
        for ent in doc.ents:
            if ent.label_ == 'DATE':
                try:
                    parsed_date = dateutil.parser.parse(ent.text)
                    standardized = parsed_date.strftime('%Y-%m-%d')
                    text = text.replace(ent.text, standardized)
                except:
                    continue
        return text

    def _process_entities(self, text: str) -> str:
        doc = self.nlp(text)
        for ent in doc.ents:
            if ent.label_ in ['GPE', 'ORG']:
                text = text.replace(ent.text, ent.text.title())
        return text

    def _normalize_whitespace(self, text: str) -> str:
        return ' '.join(text.split())

class PDFProcessor:
    def __init__(
        self,
        project_id: str = "gcp-smart-capture",
        location: str = "us",
        processor_id: str = "6d88bf439f34e6a5"
    ):
        self.logger = logging.getLogger(__name__)
        try:
            self.client = documentai.DocumentProcessorServiceClient(
                client_options=ClientOptions(
                    api_endpoint=f"{location}-documentai.googleapis.com"
                )
            )
            self.resource_name = self.client.processor_path(
                project_id, location, processor_id
            )
        except Exception as e:
            self.logger.error(f"Failed to initialize PDFProcessor: {str(e)}")
            raise

    def process_pdf(self, file_path: str) -> str:
        try:
            self.logger.info(f"Processing PDF: {file_path}")
            with open(file_path, "rb") as pdf_file:
                content = pdf_file.read()

            raw_document = documentai.RawDocument(
                content=content,
                mime_type="application/pdf"
            )
            request = documentai.ProcessRequest(
                name=self.resource_name,
                raw_document=raw_document
            )
            result = self.client.process_document(request=request)
            self.logger.info("PDF processing completed successfully")
            return result.document.text
        except FileNotFoundError:
            self.logger.error(f"PDF file not found: {file_path}")
            raise
        except Exception as e:
            self.logger.error(f"Error processing PDF: {str(e)}")
            raise


class EventExtractor:
    def __init__(self):
        self.logger = logging.getLogger(__name__)
        try:
            vertexai.init(project="gcp-smart-capture", location="us-central1")
            self.model = GenerativeModel("gemini-1.5-pro-002")
            self.generation_config = {
                "candidate_count": 1,
                "max_output_tokens": 8192,
                "temperature": 0,
                "top_p": 0.95,
            }
        except Exception as e:
            self.logger.error(f"Failed to initialize EventExtractor: {str(e)}")
            raise

    def extract_events(self, text: str) -> Union[Dict, str]:
        try:
            self.logger.info("Starting event extraction")
            prompt = f"""
            Extract events from the following text and return them in valid JSON format with this exact structure:
            {{
                "events": [
                    {{
                        "event_name": "string",
                        "location": "string",
                        "start_date": "string",
                        "end_date": "string",
                        "description": "string",
                        "event_type": "string",
                        "region": "string",
                        "owner": "string"
                    }}
                ]
            }}
            
            Text: {text}

            Description is the summary of the wwhole event 
            
            Important: Ensure the response is valid JSON that matches the exact structure above.
            """
            
            response = self.model.generate_content(
                prompt,
                generation_config=self.generation_config
            )
            
            # Get the response text
            response_text = response.text.strip()
            
            # Try to find JSON content within the response
            try:
                # Find the first { and last } in the response
                start_idx = response_text.find('{')
                end_idx = response_text.rfind('}')
                
                if start_idx != -1 and end_idx != -1:
                    json_str = response_text[start_idx:end_idx + 1]
                    return json.loads(json_str)
                else:
                    self.logger.warning("No JSON structure found in response")
                    return {"events": [], "raw_response": response_text}
                    
            except json.JSONDecodeError as e:
                self.logger.warning(f"Failed to parse JSON: {str(e)}")
                return {"events": [], "raw_response": response_text}
                
        except Exception as e:
            self.logger.error(f"Error during event extraction: {str(e)}")
            return {"events": [], "error": str(e)}

class EnhancedDocumentProcessor:
    def __init__(self, debug_mode: bool = False):
        self.logger = logging.getLogger(__name__)
        self.pdf_processor = PDFProcessor()
        self.text_preprocessor = AdvancedTextPreprocessor()
        self.event_extractor = EventExtractor()
        self.debug_mode = debug_mode

    def process_document(self, pdf_path: str) -> Dict:
        try:
            self.logger.info(f"Starting document processing for: {pdf_path}")
            
            # Extract text from PDF
            raw_text = self.pdf_processor.process_pdf(pdf_path)
            
            # Advanced preprocessing with spaCy
            processed_text = self.text_preprocessor.preprocess(raw_text, debug=self.debug_mode)
            
            # Extract events with enhanced fields
            result = self.event_extractor.extract_events(processed_text)
            
            self.logger.info("Document processing completed successfully")
            return result
        except Exception as e:
            self.logger.error(f"Error during document processing: {str(e)}")
            return {"events": [], "error": str(e)}

def main():
    logging.info("Starting main application")
    try:
        # Initialize processor with debug mode if needed
        processor = EnhancedDocumentProcessor(debug_mode=True)
        
        # Process the document
        pdf_path = r"C:\Users\Relanto\PycharmProjects\Smart_capture_Spacy\Sample_Event_Details-1-1-13.pdf"
        result = processor.process_document(pdf_path)
        
        # Print the results
        print(json.dumps(result, indent=4))
        
        # Check if any events were extracted
        if not result.get("events"):
            if "raw_response" in result:
                logging.warning("No events extracted. Raw response:")
                logging.warning(result["raw_response"])
            elif "error" in result:
                logging.error(f"Error occurred: {result['error']}")
        
        logging.info("Application completed successfully")
    except Exception as e:
        logging.error(f"Application failed: {str(e)}")
        raise

if __name__ == "__main__":
    main()

2025-01-27 20:55:01,008 - root - INFO - Starting main application
2025-01-27 20:55:02,623 - __main__ - INFO - Starting document processing for: C:\Users\Relanto\PycharmProjects\Smart_capture_Spacy\Sample_Event_Details-1-1-13.pdf
2025-01-27 20:55:02,624 - __main__ - INFO - Processing PDF: C:\Users\Relanto\PycharmProjects\Smart_capture_Spacy\Sample_Event_Details-1-1-13.pdf
2025-01-27 20:55:22,055 - __main__ - INFO - PDF processing completed successfully


Original text: Google Cloud Summit
Event
Sample Event Details
Upcoming Events - Summit Series
Summit Series
Opportunities for you to engage with experts in
live Q&As, explore demos, and gain insights
from our partne...


2025-01-27 20:55:22,639 - __main__ - INFO - Starting event extraction



Final processed text: Google Cloud Summit Event Sample Event Details Upcoming Events - Summit Series Summit Series Opportunities for you to engage with experts in live Q As, explore demos, and gain insights from our partne...


2025-01-27 20:55:50,558 - __main__ - INFO - Document processing completed successfully
2025-01-27 20:55:50,561 - root - INFO - Application completed successfully


{
    "events": [
        {
            "event_name": "Cloud Summit",
            "location": "Hong Kong",
            "start_date": "2025-05-23",
            "end_date": "2025-05-23",
            "description": null,
            "event_type": "Cloud Summit",
            "region": "Apac",
            "owner": "John"
        },
        {
            "event_name": "Cloud Summit",
            "location": "Paris",
            "start_date": "2025-05-24",
            "end_date": "2025-05-24",
            "description": null,
            "event_type": "Cloud Summit",
            "region": "EMEA",
            "owner": "Mary"
        },
        {
            "event_name": "Cloud Summit",
            "location": "Jakarta",
            "start_date": "2025-05-27",
            "end_date": "2025-05-27",
            "description": null,
            "event_type": "Cloud Summit",
            "region": "Apac",
            "owner": "Casey"
        },
        {
            "event_name": "Cloud Summit",
  